# Task 5 Fine-tune a transformer model (BERT/DistilBERT) to perform: Part-of-Speech (POS) Tagging- Chunking
# Name : Ankita Patil

In [ ]:
!pip install transformers seqeval evaluate -q

import nltk
import numpy as np
import evaluate
from nltk.corpus import treebank
from sklearn.model_selection import train_test_split

from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00


In [ ]:
nltk.download('treebank')
nltk.download('universal_tagset')

data = treebank.tagged_sents(tagset='universal')

sentences = [[w for w,t in s] for s in data]
labels = [[t for w,t in s] for s in data]

[nltk_data] Downloading package treebank to /root/nltk_data...
[nltk_data]   Package treebank is already up-to-date!
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!


In [ ]:
unique_tags = list(set(tag for sent in labels for tag in sent))

label2id = {tag:i for i,tag in enumerate(unique_tags)}
id2label = {i:tag for tag,i in label2id.items()}
num_labels = len(unique_tags)

In [ ]:
train_s, temp_s, train_l, temp_l = train_test_split(sentences, labels, test_size=0.2, random_state=42)
val_s, test_s, val_l, test_l = train_test_split(temp_s, temp_l, test_size=0.5, random_state=42)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def tokenize_and_align(sentences, labels):
    encodings = tokenizer(sentences, truncation=True, padding=True, is_split_into_words=True)
    aligned_labels = []

    for i, label in enumerate(labels):
        word_ids = encodings.word_ids(batch_index=i)
        prev = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != prev:
                label_ids.append(label2id[label[word_idx]])
            else:
                label_ids.append(-100)
            prev = word_idx

        aligned_labels.append(label_ids)

    encodings["labels"] = aligned_labels
    return encodings

In [ ]:
train_enc = tokenize_and_align(train_s, train_l)
val_enc = tokenize_and_align(val_s, val_l)
test_enc = tokenize_and_align(test_s, test_l)

In [ ]:
import torch

class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        return {k: torch.tensor(v[idx]) for k,v in self.encodings.items()}

    def __len__(self):
        return len(self.encodings["input_ids"])

In [ ]:
train_dataset = Dataset(train_enc)
val_dataset = Dataset(val_enc)
test_dataset = Dataset(test_enc)

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_preds = [
        [id2label[p] for (p,l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [id2label[l] for (p,l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_preds, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_steps=50
)

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Step,Training Loss
50,1.541942
100,0.358505
150,0.148039
200,0.103998
250,0.080451
300,0.074610
350,0.067840
400,0.064665
450,0.058098
500,0.054598


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=588, training_loss=0.2241704001718638, metrics={'train_runtime': 395.8947, 'train_samples_per_second': 23.726, 'train_steps_per_second': 1.485, 'total_flos': 1002094966241424.0, 'train_loss': 0.2241704001718638, 'epoch': 3.0})

In [ ]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DET seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NOUN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: . seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: X seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserW

{'eval_loss': 0.0574716180562973,
 'eval_precision': 0.9733333333333334,
 'eval_recall': 0.9739637305699482,
 'eval_f1': 0.973648429912593,
 'eval_accuracy': 0.9846348341923563,
 'eval_runtime': 2.6795,
 'eval_samples_per_second': 145.92,
 'eval_steps_per_second': 9.33,
 'epoch': 3.0}

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def predict(sentence):
    tokens = sentence.split()

    inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    preds = torch.argmax(outputs.logits, dim=2)[0].cpu().numpy()

    word_ids = inputs["input_ids"].cpu()
    word_ids = tokenizer(tokens, is_split_into_words=True).word_ids()

    result = []
    prev = None

    for i, word_id in enumerate(word_ids):
        if word_id is None or word_id == prev:
            continue
        result.append((tokens[word_id], id2label[preds[i]]))
        prev = word_id

    return result

predict("John works at Google in California")

[('John', 'NOUN'),
 ('works', 'VERB'),
 ('at', 'ADP'),
 ('Google', 'NOUN'),
 ('in', 'ADP'),
 ('California', 'NOUN')]

POS tagging assigns grammatical categories such as noun, verb, adjective.

Chunking identifies phrase-level structures such as noun phrases and verb phrases.

POS tagging is simpler and works at word level.

Chunking is more complex and captures higher-level linguistic patterns.

This project implements token classification using DistilBERT for POS tagging.

The NLTK Treebank dataset was used to ensure compatibility and stability.

Tokenization and label alignment were carefully handled to manage subword tokens.

The model achieved strong performance using seqeval metrics.

POS tagging was successfully implemented, and chunking was analyzed conceptually.

Challenges included handling subword alignment and maintaining label consistency.